# Lección 3.4 — Hiperparámetros del entrenamiento: tasa de aprendizaje, épocas, tamaño de lote e iteraciones

> **Módulo 3 · Cómo aprende un modelo** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 100–120 min (lección ≈ 70 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 3.3 (descenso de gradiente). Ayuda recordar el escalado de la Lección 2.3.

### 🎯 Objetivos
Al terminar esta lección podrás:
- Definir con tus palabras **tasa de aprendizaje**, **época**, **tamaño de lote** (*batch size*) e **iteración**, y calcular cuántas iteraciones hace un entrenamiento.
- Predecir qué le pasa al entrenamiento si la tasa de aprendizaje es **muy chica**, **adecuada**, **grande** o **demasiado grande**.
- Decidir **cuántas épocas** entrenar mirando la curva de pérdida.
- Ver cómo el **tamaño de lote** cambia la cantidad de pasos y el "ruido" del entrenamiento.
- Reconocer estas perillas con sus nombres en scikit-learn, Keras y PyTorch.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar ni matemáticas. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Esta vez la heladería es una **cadena** con datos de **1,000 días**. La temperatura ya viene **escalada** (media 0, desviación 1; Lección 2.3), porque en la Lección 3.3 viste que el descenso de gradiente sufre sin escalar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

plt.rcParams["figure.figsize"] = (8, 4.5)
np.seterr(all="ignore")          # cuando un entrenamiento "explota" no queremos avisos rojos, queremos verlo

rng = np.random.default_rng(42)
N = 1000
temperatura = rng.uniform(14, 36, N)
ventas = 8 * temperatura - 60 + rng.normal(0, 15, N)
X = ((temperatura - temperatura.mean()) / temperatura.std()).reshape(-1, 1)   # temperatura escalada

def mse(real, pred):
    return np.mean((pred - real) ** 2)

print(f"✅ {N} días cargados. Ventas promedio: {ventas.mean():.0f} helados")

Y esta es nuestra **máquina de entrenar**: el mismo descenso de gradiente de la Lección 3.3, pero ahora con las cuatro perillas a la vista. No hace falta que entiendas cada línea; fíjate en los comentarios.

In [ ]:
def entrenar(X, y, tasa, epocas, tam_lote, semilla=0):
    rng_ent = np.random.default_rng(semilla)
    n, p = X.shape
    w, b = np.zeros(p), 0.0                                  # perillas en cero al empezar
    perdida_epoca = [mse(y, X @ w + b)]                      # pérdida al final de cada época
    perdida_iter = [mse(y, X @ w + b)]                       # pérdida después de cada iteración
    for epoca in range(epocas):                              # 🔁 ÉPOCA: una vuelta completa a los datos
        orden = rng_ent.permutation(n)                       # barajamos los días en cada vuelta
        for inicio in range(0, n, tam_lote):                 # 📦 LOTE: un grupito de ejemplos
            lote = orden[inicio:inicio + tam_lote]
            error = X[lote] @ w + b - y[lote]
            w = w - tasa * 2 * X[lote].T @ error / len(lote)  # 👣 ITERACIÓN: un paso con la TASA
            b = b - tasa * 2 * error.mean()
            perdida_iter.append(mse(y, X @ w + b))
        perdida_epoca.append(mse(y, X @ w + b))
    return w, b, np.array(perdida_epoca), np.array(perdida_iter)

w, b, pe, pi = entrenar(X, ventas, tasa=0.1, epocas=20, tam_lote=1000)
print(f"Tras 20 épocas: w = {w[0]:.1f}, b = {b:.1f}, pérdida = {pe[-1]:.1f}")

---
## 1. 📖 La intuición: estudiar para un examen

Tienes un libro con **1,000 ejercicios resueltos** para preparar un examen.

| Concepto | En el estudio | En el entrenamiento |
|---|---|---|
| **Lote** (*batch*) | Cuántos ejercicios resuelves **antes de revisar** las respuestas y corregir tu método | Cuántos ejemplos mira el modelo antes de dar un paso |
| **Iteración** (*step*) | Cada vez que revisas y **corriges** tu método | Cada paso del descenso de gradiente |
| **Época** (*epoch*) | Una **lectura completa** del libro | Una pasada completa por **todos** los datos |
| **Tasa de aprendizaje** | **Cuánto** cambias tu método tras cada corrección: un poquito, o lo cambias todo | El tamaño del paso del excursionista |

- Si revisas tras **cada** ejercicio (lote = 1), corriges muy seguido, pero a veces te dejas llevar por un ejercicio raro.
- Si revisas tras leer **todo** el libro (lote = 1,000), cada corrección es muy informada… pero solo corriges **una vez por lectura**.
- Si tras cada corrección cambias **todo** tu método (tasa enorme), nunca te estabilizas. Si casi no lo cambias (tasa diminuta), necesitas leer el libro cien veces.

### ✋ Ejercicio 1 — Pon nombre a cada cosa
Una red neuronal se entrena así: *"Se toman 64 fotos, se calcula el error y se ajustan los pesos multiplicando el gradiente por 0.001. Se repite hasta haber usado las 50,000 fotos, y todo eso se hace 10 veces."*

¿Cuál es el **tamaño de lote**, la **tasa de aprendizaje** y el número de **épocas**?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **Tamaño de lote:** 64 fotos.
- **Tasa de aprendizaje:** 0.001.
- **Épocas:** 10 (diez pasadas completas por las 50,000 fotos).
</details>

---
## 2. 📖 El concepto: la cuenta de las iteraciones

Las cuatro perillas están conectadas por una cuenta sencilla:

$$\text{iteraciones por época} = \left\lceil \frac{\text{número de ejemplos}}{\text{tamaño de lote}} \right\rceil \qquad\qquad \text{iteraciones totales} = \text{épocas} \times \text{iteraciones por época}$$

(El símbolo ⌈ ⌉ significa "redondear hacia arriba": si sobran ejemplos, forman un último lote más pequeño.)

### 🔮 Ejercicio 2 — Calcula
Con nuestros **1,000 días**:
1. Lote de 100 y 5 épocas: ¿cuántas iteraciones (pasos) en total?
2. Lote de 32 y 1 época: ¿cuántos pasos? ¿De qué tamaño es el último lote?
3. Lote de 1,000 y 20 épocas: ¿cuántos pasos?
4. Lote de 1 y 20 épocas: ¿cuántos pasos?

✍️ *Tu respuesta:*

In [ ]:
import math

def contar_iteraciones(n, tam_lote, epocas):
    por_epoca = math.ceil(n / tam_lote)
    return por_epoca, por_epoca * epocas

for tam_lote, epocas in [(100, 5), (32, 1), (1000, 20), (1, 20)]:   # ✏️ añade tus casos
    por_epoca, total = contar_iteraciones(N, tam_lote, epocas)
    print(f"lote={tam_lote:>4}, épocas={epocas:>2} → {por_epoca:>4} pasos por época → {total:>6} pasos en total")

<details><summary><b>👀 Ver respuesta</b></summary>

1. 1,000 / 100 = 10 pasos por época × 5 = **50** pasos.
2. 1,000 / 32 = 31.25 → **32** pasos: 31 lotes de 32 y un último lote de **8** ejemplos.
3. **1** paso por época × 20 = **20** pasos.
4. **1,000** pasos por época × 20 = **20,000** pasos.

Misma cantidad de épocas, ¡**mil veces** más pasos! Esa es la gran diferencia que verás en la Sección 5 y en la Lección 3.5.
</details>

---
## 3. 📖 Perilla 1: la tasa de aprendizaje (la más importante)

Para ver solo el efecto de la tasa, usamos **todos los datos en cada paso** (lote = 1,000), así no hay ningún otro efecto mezclado.

### 🔮 Ejercicio 3 — Cuatro excursionistas
Cuatro excursionistas bajan la misma montaña con zancadas distintas: tasa **0.01**, **0.1**, **0.95** y **1.05**. Antes de ejecutar, predice para cada uno: ¿llega al fondo rápido, lento, zigzagueando o sale volando?

✍️ *Mis predicciones:* 0.01 → ___ · 0.1 → ___ · 0.95 → ___ · 1.05 → ___

In [ ]:
TASAS = [0.01, 0.1, 0.95, 1.05]       # ✏️ prueba otras
EPOCAS = 30

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
for tasa in TASAS:
    w, b, pe, _ = entrenar(X, ventas, tasa, epocas=EPOCAS, tam_lote=N)
    a1.plot(pe, "o-", ms=3, label=f"tasa = {tasa}")
    print(f"tasa = {tasa:<5} → pérdida final = {pe[-1]:,.1f}")
a1.set_yscale("log"); a1.set_ylim(100, 1e6); a1.set_xlabel("Época"); a1.set_ylabel("Pérdida (log)")
a1.set_title("Curva de pérdida según la tasa"); a1.legend(); a1.grid(alpha=.3)

# Panel derecho: cómo se mueve la perilla w en cada paso
for tasa in TASAS:
    w_vals = [entrenar(X, ventas, tasa, epocas=k, tam_lote=N)[0][0] for k in range(0, 15)]
    a2.plot(w_vals, "o-", ms=3, label=f"tasa = {tasa}")
w_fondo = entrenar(X, ventas, 0.1, epocas=200, tam_lote=N)[0][0]
a2.axhline(w_fondo, color="k", ls="--", lw=1, label="fondo del valle")
a2.set_ylim(-20, 120); a2.set_xlabel("Paso"); a2.set_ylabel("Valor de la perilla w")
a2.set_title("El camino de la perilla w"); a2.legend(); a2.grid(alpha=.3)
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **0.01 → lento**: baja de forma suave pero tras 30 épocas sigue lejísimos del fondo (≈ 6,700 contra ≈ 231).
- **0.1 → rápido y estable**: llega al fondo (≈ 231) en unas 20–25 épocas.
- **0.95 → zigzag**: `w` salta de un lado al otro del fondo (muy arriba, muy abajo…) y cada salto es algo más pequeño. Se va acercando (≈ 270 tras 30 épocas), pero de forma "nerviosa": ¡llega **más tarde** que con 0.1, aunque sus pasos son casi 10 veces más largos! Los rebotes desperdician el esfuerzo.
- **1.05 → sale volando** (*diverge*): cada salto es más grande que el anterior y la pérdida crece sin parar (≈ 6.7 millones tras 30 épocas).

En el panel derecho se ve clarísimo: la perilla `w` **se arrastra** (0.01), **llega directo** (0.1), **rebota** alrededor del fondo (0.95) o **rebota cada vez más lejos** (1.05).
</details>

### 📖 El barrido de la tasa de aprendizaje
En la práctica no se adivina la tasa: se hace un **barrido**. Entrenas con varias tasas (separadas por factores de ~3 o ~10) durante unas pocas épocas y comparas la pérdida final. La gráfica tiene casi siempre esta forma: **lenta a la izquierda, un valle de buenas tasas y un precipicio a la derecha**.

### ✏️ Ejercicio 4 — Encuentra el precipicio
Ejecuta el barrido. Luego cambia `EPOCAS_BARRIDO` a 3 y a 50, y observa cómo se mueve la zona buena.

In [ ]:
EPOCAS_BARRIDO = 10        # ✏️ prueba 3 y 50

tasas = np.logspace(-3, 0.1, 25)        # de 0.001 a ~1.26, espaciadas en escala logarítmica
finales = [entrenar(X, ventas, t, epocas=EPOCAS_BARRIDO, tam_lote=N)[2][-1] for t in tasas]

plt.plot(tasas, finales, "o-")
plt.xscale("log"); plt.yscale("log"); plt.ylim(100, 1e6)
plt.xlabel("Tasa de aprendizaje (escala log)"); plt.ylabel(f"Pérdida tras {EPOCAS_BARRIDO} épocas")
plt.title("Barrido de la tasa de aprendizaje"); plt.grid(alpha=.3); plt.show()
mejor = tasas[np.argmin(finales)]
print(f"Mejor tasa del barrido: {mejor:.3f}")

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

- Por la **izquierda** (tasas < 0.01) la pérdida queda alta: pasos demasiado cortos.
- En el **centro** hay un valle amplio de tasas buenas; la mejor de este barrido ronda **0.5**.
- Por la **derecha**, a partir de **≈ 1.0**, la pérdida se dispara: es el **precipicio**.
- Con **más épocas** (50) el valle de tasas buenas se **ensancha hacia la izquierda**: con tiempo suficiente, incluso tasas más chicas llegan. Con **pocas épocas** (3) solo las tasas cercanas al óptimo llegan.

💡 Regla práctica: elige una tasa **un poco a la izquierda del precipicio**, no justo en el borde. Cerca del borde, cualquier cambio (más datos, otro lote) puede tirarte por el precipicio.

> 📝 ¿Por qué aquí el óptimo es tan alto (0.5)? Porque nuestro valle es un **tazón perfectamente redondo** (una característica escalada). En modelos reales, como las redes neuronales, las tasas típicas son mucho más pequeñas (0.01, 0.001…). Lo que se mantiene es la **forma** del barrido.
</details>

---
## 4. 📖 Perilla 2: las épocas (¿cuánto tiempo entrenar?)

Una **época** es una pasada completa por los datos. Más épocas = más pasos = el excursionista camina más. Pero ¿cuántas hacen falta?

### 🔮 Ejercicio 5 — ¿Cuándo parar?
Con tasa 0.05 y lote = 1,000, ¿cuántas épocas crees que harán falta para llegar prácticamente al fondo? Elige: 5, 30, 300.

✍️ *Tu respuesta:*

In [ ]:
EPOCAS = 100            # ✏️
_, _, pe, _ = entrenar(X, ventas, tasa=0.05, epocas=EPOCAS, tam_lote=N)

plt.plot(pe, lw=2); plt.yscale("log")
plt.xlabel("Época"); plt.ylabel("Pérdida (log)"); plt.title("¿Cuántas épocas hacen falta?"); plt.grid(alpha=.3); plt.show()

# ¿En qué época la mejora deja de valer la pena (< 0.1 % respecto a la época anterior)?
mejora = -np.diff(pe) / pe[:-1]
estancada = np.argmax(mejora < 0.001) + 1
print(f"Pérdida en la época 5: {pe[5]:,.1f} · época 30: {pe[30]:,.1f} · época {EPOCAS}: {pe[-1]:,.1f}")
print(f"A partir de la época {estancada} la pérdida mejora menos de 0.1 % por época")

<details><summary><b>👀 Ver respuesta</b></summary>

Más cerca de **30** que de 5 o 300. En la época 5 la pérdida todavía está muy alta (≈ 7,900); en la época 30 ya casi tocó el fondo (≈ 270, contra ≈ 231 del mínimo), y a partir de la época **48** cada época mejora menos de 0.1 %: seguir entrenando es casi gastar electricidad.

- **Pocas épocas** → el excursionista se detiene a media ladera: el modelo queda **sin terminar** (subajuste).
- **Demasiadas épocas** → en este modelo simple solo pierdes tiempo, pero en modelos complejos el modelo empieza a **memorizar** los datos de entrenamiento (sobreajuste). Lo veremos en la curva de validación de la **Lección 3.7** y lo resolveremos con *early stopping* (**Lección 7.2**).
</details>

### ✋ Ejercicio 6 — Tasa y épocas van de la mano
Si **divides la tasa entre 10** (de 0.05 a 0.005), ¿cuántas épocas crees que necesitarás para llegar al mismo punto? Compruébalo cambiando la celda anterior (`tasa=0.005`, `EPOCAS=500`) y mirando en qué época se estanca.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Muchas más: la mejora baja de 0.1 % por época en la época **374** (en vez de la 48), y en la época 30 la pérdida sigue en ≈ 12,000. Con pasos 10 veces más cortos hay que dar del orden de 10 veces más pasos. Por eso la tasa y las épocas **no se eligen por separado**: si bajas una, tienes que subir la otra.
</details>

---
## 5. 📖 Perilla 3: el tamaño de lote

Hasta ahora cada paso usaba los 1,000 días. Si usamos **lotes más pequeños**, en cada época damos **más pasos**, cada uno basado en menos datos.

### 🔮 Ejercicio 7 — Misma tasa, mismas épocas, distinto lote
Entrenamos **5 épocas** con tasa 0.05 y lotes de **1,000**, **100**, **10** y **1**. Recuerda la cuenta de la Sección 2.
1. ¿Cuál llegará más abajo al final de las 5 épocas?
2. ¿Cuál tendrá la curva más "temblorosa"?

✍️ *Tu respuesta:*

In [ ]:
LOTES = [1000, 100, 10, 1]       # ✏️
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5))
for lote in LOTES:
    inicio = time.perf_counter()
    _, _, pe, pi = entrenar(X, ventas, tasa=0.05, epocas=5, tam_lote=lote)
    seg = time.perf_counter() - inicio
    a1.plot(pe, "o-", label=f"lote = {lote}")
    a2.plot(np.linspace(0, 5, len(pi)), pi, lw=1, label=f"lote = {lote}")
    print(f"lote = {lote:>4} → {len(pi) - 1:>5} pasos · pérdida final = {pe[-1]:>9,.1f} · {seg:.2f} s")
a1.set_yscale("log"); a1.set_xlabel("Época"); a1.set_ylabel("Pérdida (log)"); a1.set_title("Pérdida al final de cada época"); a1.legend(); a1.grid(alpha=.3)
a2.set_yscale("log"); a2.set_xlabel("Época"); a2.set_title("Pérdida después de cada paso"); a2.legend(); a2.grid(alpha=.3)
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. Con lote = 1,000 solo dio **5** pasos: la pérdida sigue en ≈ 7,850. Con lotes de **100** (50 pasos) y **10** (500 pasos) llega al fondo (≈ 231). ¿Y lote = 1, con 5,000 pasos? Queda en ≈ **236**, ¡un poco **peor** que lote = 10, pese a dar 10 veces más pasos!
2. El de **lote = 1**: cada paso se basa en **un solo día**, así que a veces corrige demasiado por un día raro. En el panel derecho verás que su curva **tiembla** cerca del fondo en lugar de quedarse quieta, y por eso terminó un poco más arriba.

Y fíjate en el **tiempo**: en este notebook, lote = 1 es el más lento, porque Python da los 5,000 pasos uno por uno. En una GPU, procesar un lote de 64 cuesta casi lo mismo que uno de 1: ese equilibrio entre **más pasos** y **pasos más baratos** es el tema de la **Lección 3.5**.
</details>

---
## 6. 📖 Tabla de hiperparámetros del entrenamiento

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| **Tasa de aprendizaje** | Tamaño de cada paso | Más rápido hasta cierto punto; luego **zigzag** y **divergencia** (`nan`) | Más estable, pero **lentísimo**; puede quedarse a medio camino | Haz un **barrido** (0.0001 → 1). Típico: 0.01 (SGD), 0.001 (Adam) |
| **Épocas** | Cuántas veces se recorren todos los datos | Llega más cerca del fondo; más tiempo; riesgo de **sobreajuste** en modelos complejos | Termina antes de llegar (**subajuste**) | Las necesarias para que la pérdida de **validación** deje de bajar (Lecciones 3.7 y 7.2) |
| **Tamaño de lote** | Cuántos ejemplos se usan en cada paso | Pasos más **suaves** y fiables, pero **menos pasos** por época; más memoria | Más pasos por época, pero más **ruidosos** | 32–256 (potencias de 2 por costumbre de las GPU) |
| **Iteraciones** (pasos) | Total de actualizaciones de las perillas | — (no se elige aparte) | — | Resulta de `épocas × ⌈N / lote⌉`. Algunas librerías lo fijan directamente |

### 🗺️ Los mismos nombres en distintas librerías

| Concepto | scikit-learn (`SGDRegressor`, `MLPClassifier`) | Keras | PyTorch |
|---|---|---|---|
| Tasa de aprendizaje | `eta0` · `learning_rate_init` | `optimizer=Adam(learning_rate=...)` | `torch.optim.SGD(..., lr=...)` |
| Épocas | `max_iter` (¡en estos modelos cuenta **épocas**, no pasos!) | `model.fit(..., epochs=...)` | el bucle `for epoca in range(...)` que escribes tú |
| Tamaño de lote | `batch_size` (en `MLP...`) | `model.fit(..., batch_size=...)` | `DataLoader(..., batch_size=...)` |

> 📝 Muchas librerías **bajan la tasa automáticamente** durante el entrenamiento (*learning rate schedule*): pasos largos al principio y cortos al final, como un excursionista que frena al acercarse al refugio. Lo verás en la Lección 9.6.

### ▶️ Las mismas perillas en scikit-learn
`SGDRegressor` es una regresión lineal entrenada con descenso de gradiente (estocástico, Lección 3.5). Fíjate en que sus perillas son las que ya conoces.

In [ ]:
from sklearn.linear_model import SGDRegressor

sgd = SGDRegressor(learning_rate="constant", eta0=0.01,   # tasa de aprendizaje fija
                   max_iter=20,                            # épocas
                   tol=None, random_state=0)               # tol=None: que no se detenga antes
sgd.fit(X, ventas)
print(f"scikit-learn → w = {sgd.coef_[0]:.1f}, b = {sgd.intercept_[0]:.1f}, pérdida = {mse(ventas, sgd.predict(X)):.1f}")

w, b, pe, _ = entrenar(X, ventas, tasa=0.1, epocas=50, tam_lote=N)
print(f"nuestro código → w = {w[0]:.1f}, b = {b:.1f}, pérdida = {pe[-1]:.1f}")

### ✋ Ejercicio 8
Las dos versiones llegan casi al mismo sitio. En `SGDRegressor` no hay `batch_size`: ¿qué tamaño de lote crees que usa? (Pista: mira cómo se llama: **S**tochastic **G**radient **D**escent.)

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Usa lotes de **1 ejemplo**: el descenso de gradiente **estocástico** actualiza las perillas tras cada ejemplo. Lo estudiaremos en la próxima lección, junto con sus primos (batch y mini-batch).
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma en la curva de pérdida | Causa probable | Qué probar |
|---|---|---|
| Sube y llega a `inf` / `nan` | Tasa **demasiado alta** | Dividir la tasa entre 3–10 |
| Zigzag fuerte, sube y baja sin asentarse | Tasa **un poco alta** o lote muy pequeño | Bajar la tasa; subir el lote |
| Baja en línea casi recta, sin "codo", y sigue bajando al terminar | Tasa **demasiado baja** o **pocas épocas** | Subir la tasa (barrido) o entrenar más épocas |
| Plana desde el principio | Tasa diminuta, datos sin escalar o un error en el código | Barrido de tasa; escalar; revisar el signo (Lección 3.3) |
| Cambias el lote y "deja de funcionar" | La tasa buena depende del lote | Volver a hacer el barrido de tasa tras cambiar el lote |
| Confundir épocas con iteraciones | `max_iter` significa cosas distintas según la librería | Lee la documentación y haz la cuenta `épocas × ⌈N / lote⌉` |

### 🔮 Ejercicio 9 — El diagnóstico rápido
Tu colega te muestra que su pérdida va así, época por época: `22,090 → 22,002 → 21,915 → 21,829 → 21,742 …` (baja unos 87 por época, siempre casi igual). Lleva 10 épocas. ¿Qué le recomiendas? Comprueba tu idea con la celda (es una simulación de su caso).

✍️ *Tu respuesta:*

In [ ]:
_, _, pe_colega, _ = entrenar(X, ventas, tasa=0.001, epocas=10, tam_lote=N)
print("Colega (tasa 0.001):", np.round(pe_colega[:6]).astype(int), "…", round(pe_colega[-1]))

TASA_SUGERIDA = 0.001          # ✏️ cambia este valor con tu recomendación
_, _, pe_tuya, _ = entrenar(X, ventas, tasa=TASA_SUGERIDA, epocas=10, tam_lote=N)
print(f"Tu sugerencia (tasa {TASA_SUGERIDA}): pérdida tras 10 épocas = {pe_tuya[-1]:,.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Bajadas pequeñas y **constantes**, sin codo, son la firma de una **tasa demasiado baja**: el excursionista avanza a pasos de hormiga. Recomendación: **subir la tasa** (×10 o ×100) en vez de entrenar 1,000 épocas. Con `0.01` la pérdida tras 10 épocas baja a ≈ 14,800; con `0.1`, a ≈ 480; y con `0.3` ya llega al fondo (≈ 231).
</details>

---
## 8. 🧠 Resumen en 5 puntos
1. **Época** = una pasada por todos los datos · **lote** = ejemplos por paso · **iteración** = un paso · `iteraciones = épocas × ⌈N / lote⌉`.
2. La **tasa de aprendizaje** es la perilla más importante: muy baja = lento; muy alta = zigzag o `nan`. Se elige con un **barrido**, un poco antes del precipicio.
3. Las **épocas** se eligen mirando la curva: cuando la pérdida (de validación, Lección 3.7) deja de bajar, para.
4. Lotes **pequeños** = más pasos por época pero más ruidosos; lotes **grandes** = pasos suaves pero pocos.
5. Las perillas están **conectadas**: si bajas la tasa necesitas más épocas; si cambias el lote, revisa la tasa.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** Tienes 60,000 imágenes, lote de 100 y 3 épocas. ¿Cuántas iteraciones hace el entrenamiento?
- a) 300 · b) 600 · c) 1,800 · d) 180,000

**2.** La pérdida pasa de 0.7 a 5.2, luego a 380, luego a `nan`. ¿Qué perilla tocas primero?
- a) Subir épocas · b) Bajar la tasa de aprendizaje · c) Subir la tasa · d) Bajar el lote a 1

**3.** Con lote = tamaño del conjunto completo, ¿cuántas iteraciones hay por época?
- a) Una · b) Tantas como ejemplos · c) Depende de la tasa · d) Cero

**4.** Si divides la tasa de aprendizaje entre 10 y no cambias nada más, lo más probable es que…
- a) el modelo llegue al mismo punto en el mismo número de épocas · b) necesite unas 10 veces más épocas · c) diverja · d) la pérdida final sea 10 veces menor

**5.** ¿Cuál de estos **no** es un hiperparámetro del entrenamiento?
- a) Tasa de aprendizaje · b) Tamaño de lote · c) El peso `w` · d) Número de épocas

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** 60,000 / 100 = 600 por época × 3 = **1,800**.
2. **b)** Una pérdida que explota es el síntoma clásico de una tasa demasiado alta.
3. **a)** Un único paso con todos los datos.
4. **b)** Pasos 10 veces más cortos → unas 10 veces más pasos.
5. **c)** `w` es un **parámetro**: lo aprende el modelo (Lección 3.1).
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Tu jefe dice: *"Pon 1,000 épocas, así seguro aprende más"*. ¿Qué le respondes? Da dos razones.

**B.** Un compañero encontró que tasa = 0.1 funciona perfecto con lote = 256. Cambia a lote = 8 para ahorrar memoria, deja la tasa igual y ahora la pérdida tiembla muchísimo y a veces sube. ¿Qué pasó y qué le sugieres?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** 1) Cuando la pérdida deja de bajar, más épocas solo **gastan tiempo y dinero** (cómputo). 2) En modelos flexibles, entrenar de más hace que el modelo **memorice** los datos de entrenamiento y empeore con datos nuevos (sobreajuste). La forma correcta es mirar la **curva de validación** y parar cuando deja de mejorar (*early stopping*, Lecciones 3.7 y 7.2).

**B.** Con lotes de 8, cada paso se calcula con muy pocos ejemplos: la dirección es **ruidosa**. Con la misma tasa, esos pasos ruidosos son demasiado largos. Sugerencia: **bajar la tasa** (por ejemplo a 0.01–0.03) y repetir el barrido. Regla general: **si cambias el lote, revisa la tasa**.
</details>

### 9.3 🧩 Práctica integradora: entrena el pronóstico de la cadena 🍦

La cadena quiere predecir las ventas con **tres** características: temperatura (°C), si es fin de semana (0/1) y lluvia (mm). Tu jefe te da un **presupuesto**: el entrenamiento debe terminar en **10 épocas como máximo**.

**Paso 1 ▶️ — Los datos (sin escalar)**

In [ ]:
rng3 = np.random.default_rng(7)
temp3 = rng3.uniform(14, 36, N)
finde3 = rng3.integers(0, 2, N)
lluvia3 = rng3.exponential(5, N)
ventas3 = 7 * temp3 + 40 * finde3 - 3 * lluvia3 - 30 + rng3.normal(0, 15, N)

X3_crudo = np.column_stack([temp3, finde3, lluvia3])
print("Rango de cada columna (mín – máx):")
for nombre, col in zip(["temperatura", "fin de semana", "lluvia"], X3_crudo.T):
    print(f"   {nombre:<14} {col.min():6.1f} – {col.max():6.1f}")

**Paso 2 🔮 — Primer intento.** Tu compañero entrenó con los datos **sin escalar**, tasa 0.01, lote 32, 10 épocas. ¿Qué crees que pasó?

✍️ *Mi predicción:*

In [ ]:
_, _, pe3, _ = entrenar(X3_crudo, ventas3, tasa=0.01, epocas=10, tam_lote=32)
print("Pérdida por época:", pe3[:4], "…")

<details><summary><b>👀 Ver respuesta</b></summary>

La pérdida **explota** a `inf` o `nan` en la primera época. La temperatura y la lluvia tienen valores grandes, el valle es un cañón empinadísimo (Lección 3.3) y una tasa de 0.01 es demasiado grande para él.
</details>

**Paso 3 ✏️ — Escala.** Escala las tres columnas (restar la media y dividir por la desviación, Lección 2.3) y repite **exactamente** el mismo entrenamiento.

In [ ]:
X3 = (X3_crudo - X3_crudo.mean(axis=0)) / X3_crudo.std(axis=0)      # ✏️ (ya está hecho; entiéndelo)

_, _, pe3, _ = entrenar(X3, ventas3, tasa=0.01, epocas=10, tam_lote=32)
print(f"Pérdida al final de cada época: {np.round(pe3[1:], 1)}")

<details><summary><b>👀 Ver respuesta</b></summary>

Ahora sí baja, época a época: 6,922 → 2,069 → 728 → … → **≈ 221** en la décima. Escalar convirtió un problema imposible en uno fácil sin tocar ninguna otra perilla.
</details>

**Paso 4 ▶️ — La meta.** ¿Cuál es la mejor pérdida posible con este modelo? `LinearRegression` la calcula con una fórmula directa (sin descenso de gradiente).

In [ ]:
from sklearn.linear_model import LinearRegression
meta = mse(ventas3, LinearRegression().fit(X3, ventas3).predict(X3))
print(f"Pérdida mínima posible: {meta:.1f}")

**Paso 5 ✏️ — Ajusta tasa y lote dentro del presupuesto.** Busca una combinación de `TASA` y `LOTE` que, en **≤ 10 épocas**, deje la pérdida a menos de **1 %** de la meta. Prueba al menos: lote 1,000 con varias tasas, y lote 32 con varias tasas.

In [ ]:
TASA, LOTE, EPOCAS = 0.01, 1000, 10          # ✏️

_, _, pe3, _ = entrenar(X3, ventas3, tasa=TASA, epocas=EPOCAS, tam_lote=LOTE)
pasos = contar_iteraciones(N, LOTE, EPOCAS)[1]
print(f"tasa={TASA}, lote={LOTE}, {EPOCAS} épocas ({pasos} pasos) → pérdida = {pe3[-1]:,.1f}  "
      f"({(pe3[-1] / meta - 1):+.1%} respecto a la meta)")

<details><summary><b>👀 Ver respuesta</b></summary>

- Lote **1,000** (solo **10 pasos**): con tasa 0.01 la pérdida sigue altísima (≈ 16,800) y con 0.1 todavía queda un 129 % por encima de la meta. Hace falta una tasa grande: con **0.3** o **0.5** llega a la meta, y con **1.0** diverge.
- Lote **32** (**320 pasos**): con tasa **0.01** o **0.03** queda a menos de 0.1 % de la meta, y con 0.1 a un 0.2 %. Con tasas altas el ruido del lote pequeño la deja temblando por encima: +3.6 % con 0.3, +10 % con 0.5, y con 1.0 explota.

Cualquier combinación que quede a < 1 % es válida. Lo importante: **con lote completo necesitas pasos grandes; con lotes pequeños, pasos más chicos y muchos más**.
</details>

**Paso 6 ✋ — Explícaselo a tu jefe.** En 3 frases y sin tecnicismos: ¿qué configuración elegiste y por qué?

✍️ *Mi explicación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo: *"Primero pusimos todas las variables en la misma escala; sin eso el entrenamiento se rompía. Después probamos varios tamaños de paso y elegimos uno que llega a la mejor precisión posible en menos de 10 vueltas a los datos, sin quedar al borde de volverse inestable. El modelo final se equivoca en promedio unos 15 helados por día, que es el ruido natural de las ventas."*

(El RMSE ≈ √221 ≈ 15 helados: el ruido con el que se generaron los datos.)
</details>

---
## 📚 Para profundizar
- 📘 **Lectura:** Google, *Machine Learning Crash Course*, sección *Linear regression: Hyperparameters* (tiene un ejercicio interactivo de tasa de aprendizaje).
- 📺 **Video:** Andrew Ng, *"Mini-batch gradient descent"* y *"Learning rate"* (curso *Improving Deep Neural Networks*, inglés con subtítulos).
- 📘 **Artículo:** Leslie Smith, *"Cyclical Learning Rates for Training Neural Networks"* (2017): el origen del barrido de la tasa (*LR range test*).

---
⬅️ **Anterior:** [3.3 Optimización: el excursionista en la niebla (descenso de gradiente)](3.3-descenso-de-gradiente.ipynb) · ➡️ **Siguiente:** [3.5 Variantes: batch, estocástico y mini-batch](3.5-batch-estocastico-minibatch.ipynb)